In [31]:
import pandas as pd
import numpy as np

df = pd.read_csv("D:/Visual Studio Code/Python/quant-workflow/data/export.csv")
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 341037 entries, 0 to 341036
Data columns (total 9 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   YEAR              341037 non-null  int64  
 1   MONTH             341037 non-null  int64  
 2   SUPPLIER          340833 non-null  str    
 3   ITEM CODE         341037 non-null  str    
 4   ITEM DESCRIPTION  341037 non-null  str    
 5   ITEM TYPE         341037 non-null  str    
 6   RETAIL SALES      341031 non-null  float64
 7   RETAIL TRANSFERS  341037 non-null  float64
 8   WAREHOUSE SALES   341037 non-null  float64
dtypes: float64(3), int64(2), str(4)
memory usage: 23.4 MB


Task 1: Operational Shrinkage & Inventory Leakage Detection

Mendeteksi kerugian/kebocoran arus barang dengan membandingkan barang yang keluar dari gudang (WAREHOUSE SALES) terhadap penjualan toko dan transfer (RETAIL SALES + RETAIL TRANSFERS).

Statistik Anomali: Menggunakan Z-Score pada Net Drift untuk mendeteksi SUPPLIER atau ITEM CODE dengan deviasi ekstrim (indikasi fraud, loss, atau barang rusak di jalan).


In [32]:
df["NET DRIFT"] = df["WAREHOUSE SALES"] - (df["RETAIL SALES"] + df["RETAIL TRANSFERS"])

supplier_risk = (
    df.groupby("SUPPLIER")
    .agg(
        total_warehouse=("WAREHOUSE SALES", "sum"),
        total_retail=("RETAIL SALES", "sum"),
        total_transfers=("RETAIL TRANSFERS", "sum"),
        total_leakage=("NET DRIFT", "sum"),
        jumlah_transaksi=("ITEM CODE", "count"),
    )
    .reset_index()
)

supplier_risk["LEAKAGE RATIO %"] = (
    supplier_risk["total_leakage"] / supplier_risk["total_warehouse"]
) * 100

mean_leak = supplier_risk["total_leakage"].mean()
std_leak = supplier_risk["total_leakage"].std()
supplier_risk["Z_SCORE_LEAKAGE"] = (
    supplier_risk["total_leakage"] - mean_leak
) / std_leak

high_risk_suppliers = supplier_risk[supplier_risk["Z_SCORE_LEAKAGE"] > 3].sort_values(
    by="total_leakage", ascending=False
)

print("=== TOP SUPPLIER BERISIKO TINGGI (HIGH SHRINKAGE/LEAKAGE Z-SCORE > 3) ===")
print(
    high_risk_suppliers[
        [
            "SUPPLIER",
            "total_warehouse",
            "total_leakage",
            "LEAKAGE RATIO %",
            "Z_SCORE_LEAKAGE",
        ]
    ].head(10)
)

=== TOP SUPPLIER BERISIKO TINGGI (HIGH SHRINKAGE/LEAKAGE Z-SCORE > 3) ===
                   SUPPLIER  total_warehouse  total_leakage  LEAKAGE RATIO %  \
113           CROWN IMPORTS       1917246.02     1727689.72        90.113095   
300  MILLER BREWING COMPANY       1559556.74     1373001.10        88.037906   
22       ANHEUSER BUSCH INC       1467978.86     1221541.23        83.212454   
223            HEINEKEN USA        904228.32      784692.42        86.780341   

     Z_SCORE_LEAKAGE  
113        14.899238  
300        11.827294  
22         10.515506  
223         6.731976  


In [33]:
# 1. Agregasi Penjualan Harian/Bulanan per ITEM CODE untuk Mengukur Rata-Rata & Volatilitas (Std Dev)
item_stats = (
    df.groupby(["ITEM CODE", "ITEM DESCRIPTION", "ITEM TYPE"])
    .agg(
        avg_sales=("RETAIL SALES", "mean"),
        std_sales=("RETAIL SALES", "std"),
        total_sales=("RETAIL SALES", "sum"),
    )
    .reset_index()
    .fillna(0)
)

# 2. Parameter Risiko Inventory
Z_service_level = 1.645  # Target Service Level 95%
lead_time_days = 7  # Asumsi Waktu Tunggu Pasokan 7 Hari

# 3. Hitung Safety Stock & Reorder Point (ROP) per Item
item_stats["SAFETY_STOCK"] = (
    Z_service_level * item_stats["std_sales"] * np.sqrt(lead_time_days)
)
item_stats["REORDER_POINT"] = (item_stats["avg_sales"] * lead_time_days) + item_stats[
    "SAFETY_STOCK"
]

# Filter Item dengan Volatilitas Tinggi (Risiko Stockout Paling Besar)
high_stockout_risk_items = item_stats.sort_values(by="SAFETY_STOCK", ascending=False)

print(
    "=== TOP 10 ITEM DENGAN RISIKO STOCKOUT HIGHEST (REQUIRED SAFETY STOCK TERBESAR) ==="
)
print(
    high_stockout_risk_items[
        [
            "ITEM CODE",
            "ITEM DESCRIPTION",
            "avg_sales",
            "std_sales",
            "SAFETY_STOCK",
            "REORDER_POINT",
        ]
    ].head(10)
)

=== TOP 10 ITEM DENGAN RISIKO STOCKOUT HIGHEST (REQUIRED SAFETY STOCK TERBESAR) ===
      ITEM CODE                              ITEM DESCRIPTION    avg_sales  \
4973          2                                           ICE  1635.000000   
43560     96741                    CORONA EXTRA 4/6 NR - 12OZ   323.292222   
32662     53929                 TITO'S HANDMADE VODKA - 1.75L  1153.420741   
43706     98744                 STELLA ARTOIS 4/6 NR - 11.2OZ   274.912963   
43561     96750                   CORONA EXTRA 2/12 NR - 12OZ   388.266296   
32079     50744  WHITE CLAW VARIETY PACK #1  2/12PK 12.OZ CAN   314.096667   
37467     69573                 TRULY VARIETY 24PK LOOSE CANS   394.500000   
43491     96083                        HEINEKEN 4/6 NR - 12OZ   243.662222   
43660     97918                STELLA ARTOIS 2/12 NR - 11.2OZ   308.714815   
11984      2909                 YUENGLING LAGER 4/6 NR - 12OZ   193.549630   

        std_sales  SAFETY_STOCK  REORDER_POINT  
4973   9

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# 1. Ambil data dari task sebelumnya
top_suppliers_plot = high_risk_suppliers.head(5).sort_values(
    by="Z_SCORE_LEAKAGE", ascending=True
)
top_items_plot = high_stockout_risk_items.head(10).sort_values(
    by="SAFETY_STOCK", ascending=True
)

# 2. Buat Subplots Dashboard
fig = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=(
        "Top 5 Supplier Leakage Z-Score (Operational Risk Anomaly)",
        "Top 10 Item Safety Stock vs Avg Sales (Stockout Risk)",
    ),
    horizontal_spacing=0.15,
)

# Grafik Kiri: Anomali Z-Score Kebocoran Supplier
fig.add_trace(
    go.Bar(
        x=top_suppliers_plot["Z_SCORE_LEAKAGE"],
        y=top_suppliers_plot["SUPPLIER"],
        orientation="h",
        marker=dict(
            color=top_suppliers_plot["Z_SCORE_LEAKAGE"],
            colorscale="Reds",
            showscale=False,
        ),
        text=top_suppliers_plot["Z_SCORE_LEAKAGE"].round(2),
        textposition="outside",
        name="Z-Score Leakage",
    ),
    row=1,
    col=1,
)

# Garis Ambang Batas Anomali Z = 3
fig.add_vline(
    x=3,
    line_dash="dash",
    line_color="black",
    annotation_text="Threshold Anomali (Z = 3)",
    annotation_position="bottom right",
    row=1,
    col=1,
)

# Grafik Kanan: Perbandingan Sales vs Safety Stock
fig.add_trace(
    go.Bar(
        x=top_items_plot["avg_sales"],
        y=top_items_plot["ITEM DESCRIPTION"],
        orientation="h",
        name="Rata-Rata Penjualan Harian",
        marker_color="lightblue",
    ),
    row=1,
    col=2,
)

fig.add_trace(
    go.Bar(
        x=top_items_plot["SAFETY_STOCK"],
        y=top_items_plot["ITEM DESCRIPTION"],
        orientation="h",
        name="Safety Stock Diperlukan (Level 95%)",
        marker_color="crimson",
    ),
    row=1,
    col=2,
)

# Formatting Layout
fig.update_layout(
    title_text="<b>SUPPLY CHAIN & OPERATIONAL RISK DASHBOARD</b>",
    height=550,
    width=1200,
    barmode="group",
    showlegend=True,
    legend=dict(
        orientation="h", yanchor="bottom", y=-0.2, xanchor="center", x=0.5
    ),  # Perbaikan: 'orientation'
    template="plotly_white",
)

fig.update_xaxes(title_text="Z-Score Deviasi", row=1, col=1)
fig.update_xaxes(title_text="Volume Unit", row=1, col=2)

fig.show()

In [36]:
summary_text = f"""
================================================================================
          EXECUTIVE SUMMARY: OPERATIONAL & INVENTORY RISK MITIGATION
================================================================================

1. TEMUAN UTAMA KEBOCORAN PASOKAN (OPERATIONAL SHRINKAGE / LEAKAGE):
   - Indikasi kebocoran pasokan paling ekstrem terdeteksi pada vendor 'CROWN IMPORTS'
     dengan total Net Drift sebesar 1.727.689,72 unit (Leakage Ratio: 90,11%).
   - Nilai Z-Score anomali mencapai 14.90 (Sangat jauh melampaui batas aman Z > 3).
   - REKOMENDASI MITIGASI:
     * Lakukan audit fisik (Stock Opname) serta rekonsiliasi data pengiriman dari 
       gudang utama ke toko untuk vendor CROWN IMPORTS, MILLER BREWING, dan ANHEUSER BUSCH.
     * Periksa potensi kerugian fisik (breakage/damage) pada kemasan botol kaca
       serta pembaharuan 'item mapping' satuan (Unit of Measure) sistem.

2. TEMUAN RISIKO STOCKOUT (INVENTORY SAFETY BUFFER):
   - Fluktuasi penjualan paling volatil tercatat pada produk 'ICE' (Code: 2) dengan
     standar deviasi harian (std) sebesar 997,13 unit.
   - Untuk menjaga Service Level 95% (Z = 1,645) pada Lead Time 7 hari, diperlukan
     Safety Stock sebesar 4.339,76 unit dan Reorder Point (ROP) di angka 15.784,76 unit.
   - REKOMENDASI MITIGASI:
     * Otomatiskan Purchase Order (PO) di sistem WMS/ERP ketika persediaan ICE 
       menyentuh angka ROP 15.785 unit.
     * Evaluasi negoisasi Lead Time dengan supplier untuk memangkas Holding Cost 
       pada produk-produk volatil FMCG (Corona, Tito's Vodka, Heineken).
================================================================================
"""

print(summary_text)


          EXECUTIVE SUMMARY: OPERATIONAL & INVENTORY RISK MITIGATION

1. TEMUAN UTAMA KEBOCORAN PASOKAN (OPERATIONAL SHRINKAGE / LEAKAGE):
   - Indikasi kebocoran pasokan paling ekstrem terdeteksi pada vendor 'CROWN IMPORTS'
     dengan total Net Drift sebesar 1.727.689,72 unit (Leakage Ratio: 90,11%).
   - Nilai Z-Score anomali mencapai 14.90 (Sangat jauh melampaui batas aman Z > 3).
   - REKOMENDASI MITIGASI:
     * Lakukan audit fisik (Stock Opname) serta rekonsiliasi data pengiriman dari 
       gudang utama ke toko untuk vendor CROWN IMPORTS, MILLER BREWING, dan ANHEUSER BUSCH.
     * Periksa potensi kerugian fisik (breakage/damage) pada kemasan botol kaca
       serta pembaharuan 'item mapping' satuan (Unit of Measure) sistem.

2. TEMUAN RISIKO STOCKOUT (INVENTORY SAFETY BUFFER):
   - Fluktuasi penjualan paling volatil tercatat pada produk 'ICE' (Code: 2) dengan
     standar deviasi harian (std) sebesar 997,13 unit.
   - Untuk menjaga Service Level 95% (Z = 1,645) pada Lead Time